In [1]:
import pandas as pd
import numpy as np
import random
from faker import Faker
from datetime import datetime, timedelta
random.seed(42)

In [2]:
loan_data = pd.read_csv("/Users/aishwarya/Desktop/Projects/loan-portfolio-risk-analyzer/data/processed/loans.csv")
loan_data

,customer_id,loan_id,name,age,state,income_bracket,credit_score_band,score,grade,term_months,origination_date,apr,loan_amount,sheduled_payments,scheduled_payments,vintage_month
0,c2a7af9e-ab79-4005-add1-77d2c700d84c,L1001,Christopher Miller,99,Maine,90k+,Very Good,778,A,36,2023-05-03,0.075,26000,808.76,808.76,2023-05-01
1,101cd468-5a5c-4689-a4c3-8c4119613698,L1002,Jason Nguyen,32,Texas,60k-90k,Excellent,805,A,36,2023-09-14,0.075,13500,419.93,419.93,2023-09-01
2,fa900d46-88e1-4883-aa6f-30c077c8cd55,L1003,Michael Monroe,21,Rhode Island,<30k,Good,706,C,36,2024-09-26,0.139,20500,699.65,699.65,2024-09-01
3,38c92c57-215c-48da-affe-6666fa72766d,L1004,Jennifer Martin,53,Oklahoma,90k+,Good,726,B,36,2024-01-29,0.105,26500,861.31,861.31,2024-01-01
4,ade19b1a-5d3e-438b-bd9e-f14dabd69cb9,L1005,David Snyder,49,Hawaii,30k-60k,Very Good,764,A,36,2024-07-23,0.075,8000,248.85,248.85,2024-07-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1219,baad3a54-d2c2-4533-8116-99f783761819,L2220,Michael Gregory,70,Washington,60k-90k,Very Good,758,B,36,2024-01-28,0.105,18000,585.04,585.04,2024-01-01
1220,2ce55216-dd6b-483a-9aec-ce201c8a3085,L2221,Stefanie Warren,29,West Virginia,90k+,Very Good,774,A,36,2023-12-21,0.075,6000,186.64,186.64,2023-12-01
1221,a98d00e2-7d55-4459-b168-b9075a8d5640,L2222,Melvin Pearson,46,Missouri,60k-90k,Very Good,784,A,36,2024-02-20,0.075,7000,217.74,217.74,2024-02-01
1222,b198002d-13de-4f22-8b81-457e3807a733,L2223,Nicole Meza,32,Rhode Island,30k-60k,Good,732,B,36,2024-02-25,0.105,37500,1218.84,1218.84,2024-02-01


In [3]:
loan_data.dtypes

customer_id            object
loan_id                object
name                   object
age                     int64
state                  object
income_bracket         object
credit_score_band      object
score                   int64
grade                  object
term_months             int64
origination_date       object
apr                   float64
loan_amount             int64
sheduled_payments     float64
scheduled_payments    float64
vintage_month          object
dtype: object

In [4]:
loan_data['origination_date'] = pd.to_datetime(loan_data['origination_date'])
loan_data['vintage_month'] = pd.to_datetime(loan_data['vintage_month'])
loan_data.dtypes

customer_id                   object
loan_id                       object
name                          object
age                            int64
state                         object
income_bracket                object
credit_score_band             object
score                          int64
grade                         object
term_months                    int64
origination_date      datetime64[ns]
apr                          float64
loan_amount                    int64
sheduled_payments            float64
scheduled_payments           float64
vintage_month         datetime64[ns]
dtype: object

In [5]:
loan_data = loan_data.drop(columns=['sheduled_payments']) 

In [6]:
loan_data

,customer_id,loan_id,name,age,state,income_bracket,credit_score_band,score,grade,term_months,origination_date,apr,loan_amount,scheduled_payments,vintage_month
0,c2a7af9e-ab79-4005-add1-77d2c700d84c,L1001,Christopher Miller,99,Maine,90k+,Very Good,778,A,36,2023-05-03,0.075,26000,808.76,2023-05-01
1,101cd468-5a5c-4689-a4c3-8c4119613698,L1002,Jason Nguyen,32,Texas,60k-90k,Excellent,805,A,36,2023-09-14,0.075,13500,419.93,2023-09-01
2,fa900d46-88e1-4883-aa6f-30c077c8cd55,L1003,Michael Monroe,21,Rhode Island,<30k,Good,706,C,36,2024-09-26,0.139,20500,699.65,2024-09-01
3,38c92c57-215c-48da-affe-6666fa72766d,L1004,Jennifer Martin,53,Oklahoma,90k+,Good,726,B,36,2024-01-29,0.105,26500,861.31,2024-01-01
4,ade19b1a-5d3e-438b-bd9e-f14dabd69cb9,L1005,David Snyder,49,Hawaii,30k-60k,Very Good,764,A,36,2024-07-23,0.075,8000,248.85,2024-07-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1219,baad3a54-d2c2-4533-8116-99f783761819,L2220,Michael Gregory,70,Washington,60k-90k,Very Good,758,B,36,2024-01-28,0.105,18000,585.04,2024-01-01
1220,2ce55216-dd6b-483a-9aec-ce201c8a3085,L2221,Stefanie Warren,29,West Virginia,90k+,Very Good,774,A,36,2023-12-21,0.075,6000,186.64,2023-12-01
1221,a98d00e2-7d55-4459-b168-b9075a8d5640,L2222,Melvin Pearson,46,Missouri,60k-90k,Very Good,784,A,36,2024-02-20,0.075,7000,217.74,2024-02-01
1222,b198002d-13de-4f22-8b81-457e3807a733,L2223,Nicole Meza,32,Rhode Island,30k-60k,Good,732,B,36,2024-02-25,0.105,37500,1218.84,2024-02-01


In [7]:
rung = {}
balance = {}

for r,v in zip(loan_data['loan_id'],loan_data['loan_amount']):
    rung[r] = 0
    balance[r] = v


In [8]:
dead_loans = set()
charged_off = set()
miss_probability = {'A':0.004,'B':0.008,'C':0.011,'D':0.018,'E':0.028,'F':0.037,'G':0.057}
cure_probability = {1:0.55,2:0.30,3:0.15}
snapshots = []


for month in pd.date_range(start= loan_data['origination_date'].min(), end='2027-12-31', freq='MS'):
    for index, row in loan_data.iterrows():
        if row['origination_date'] > month:
            continue
        elif row['loan_id'] in dead_loans:
            continue
        else: 
            current_rung = rung[row['loan_id']]
            if current_rung == 0:
                mis_prob = miss_probability[row['grade']]
                roll = random.random()
                if roll < mis_prob:
                    rung[row['loan_id']] = 1
                else:
                    balance[row['loan_id']] = balance[row['loan_id']] - row['scheduled_payments']
                    if balance[row['loan_id']] <= 0 or month >= row['origination_date'] +pd.DateOffset(months=36):
                        dead_loans.add(row['loan_id'])
            elif current_rung == 1:
                cure = cure_probability[current_rung]
                roll = random.random()
                if roll < cure:
                    rung[row['loan_id']] = 0
                else:
                    rung[row['loan_id']] += 1
            elif current_rung == 2:
                cure = cure_probability[current_rung]
                roll = random.random()
                if roll < cure:
                    rung[row['loan_id']] = 0
                else:
                    rung[row['loan_id']] += 1
            elif current_rung == 3:
                cure = cure_probability[current_rung]
                roll = random.random()
                if roll < cure:
                    rung[row['loan_id']] = 0
                else:
                    rung[row['loan_id']] +=1
            elif current_rung == 4:
                dead_loans.add(row['loan_id'])
                charged_off.add(row['loan_id'])
       
            snapshots.append({'loan_id':row['loan_id'],'month':month,'rung':rung[row['loan_id']],'balance':balance[row['loan_id']]})
       

In [9]:
balance['L1005']

-212.05000000000433

In [10]:
loan_data[loan_data['loan_id'] == 'L1005']

,customer_id,loan_id,name,age,state,income_bracket,credit_score_band,score,grade,term_months,origination_date,apr,loan_amount,scheduled_payments,vintage_month
4,ade19b1a-5d3e-438b-bd9e-f14dabd69cb9,L1005,David Snyder,49,Hawaii,30k-60k,Very Good,764,A,36,2024-07-23,0.075,8000,248.85,2024-07-01


In [11]:
status_monthly = pd.DataFrame(snapshots)
status_monthly

,loan_id,month,rung,balance
0,L1129,2023-01-01,0,7266.70
1,L2206,2023-01-01,0,29459.06
2,L1045,2023-02-01,0,31004.60
3,L1055,2023-02-01,0,11609.97
4,L1082,2023-02-01,0,10158.72
...,...,...,...,...
36780,L1879,2027-11-01,0,-623.00
36781,L1912,2027-11-01,0,-702.23
36782,L2003,2027-11-01,0,-22.81
36783,L2044,2027-11-01,0,-185.42


In [12]:
len(charged_off)/len(loan_data)

0.09395424836601307

In [13]:
loan_data['charged_off'] = loan_data['loan_id'].isin(charged_off)
loan_data

,customer_id,loan_id,name,age,state,income_bracket,credit_score_band,score,grade,term_months,origination_date,apr,loan_amount,scheduled_payments,vintage_month,charged_off
0,c2a7af9e-ab79-4005-add1-77d2c700d84c,L1001,Christopher Miller,99,Maine,90k+,Very Good,778,A,36,2023-05-03,0.075,26000,808.76,2023-05-01,False
1,101cd468-5a5c-4689-a4c3-8c4119613698,L1002,Jason Nguyen,32,Texas,60k-90k,Excellent,805,A,36,2023-09-14,0.075,13500,419.93,2023-09-01,False
2,fa900d46-88e1-4883-aa6f-30c077c8cd55,L1003,Michael Monroe,21,Rhode Island,<30k,Good,706,C,36,2024-09-26,0.139,20500,699.65,2024-09-01,False
3,38c92c57-215c-48da-affe-6666fa72766d,L1004,Jennifer Martin,53,Oklahoma,90k+,Good,726,B,36,2024-01-29,0.105,26500,861.31,2024-01-01,False
4,ade19b1a-5d3e-438b-bd9e-f14dabd69cb9,L1005,David Snyder,49,Hawaii,30k-60k,Very Good,764,A,36,2024-07-23,0.075,8000,248.85,2024-07-01,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1219,baad3a54-d2c2-4533-8116-99f783761819,L2220,Michael Gregory,70,Washington,60k-90k,Very Good,758,B,36,2024-01-28,0.105,18000,585.04,2024-01-01,True
1220,2ce55216-dd6b-483a-9aec-ce201c8a3085,L2221,Stefanie Warren,29,West Virginia,90k+,Very Good,774,A,36,2023-12-21,0.075,6000,186.64,2023-12-01,False
1221,a98d00e2-7d55-4459-b168-b9075a8d5640,L2222,Melvin Pearson,46,Missouri,60k-90k,Very Good,784,A,36,2024-02-20,0.075,7000,217.74,2024-02-01,False
1222,b198002d-13de-4f22-8b81-457e3807a733,L2223,Nicole Meza,32,Rhode Island,30k-60k,Good,732,B,36,2024-02-25,0.105,37500,1218.84,2024-02-01,True


In [14]:
loan_data.groupby('grade')['charged_off'].mean()

grade
A    0.047228
B    0.063492
C    0.091503
D    0.096000
E    0.254545
F    0.218750
G    0.250000
Name: charged_off, dtype: float64

In [15]:
loan_data['grade'].value_counts()

A    487
B    252
C    153
D    125
G     88
F     64
E     55
Name: grade, dtype: int64

In [16]:
rung_labels = {0:'current', 1:'30 days late', 2:'60 days late', 3:'90 days late', 4:'120 days late'}

status_monthly['status'] = status_monthly['rung'].map(rung_labels)

In [17]:
status_monthly

,loan_id,month,rung,balance,status
0,L1129,2023-01-01,0,7266.70,current
1,L2206,2023-01-01,0,29459.06,current
2,L1045,2023-02-01,0,31004.60,current
3,L1055,2023-02-01,0,11609.97,current
4,L1082,2023-02-01,0,10158.72,current
...,...,...,...,...,...
36780,L1879,2027-11-01,0,-623.00,current
36781,L1912,2027-11-01,0,-702.23,current
36782,L2003,2027-11-01,0,-22.81,current
36783,L2044,2027-11-01,0,-185.42,current


In [18]:
charge_offs = status_monthly[status_monthly['loan_id'].isin(charged_off)]
charge_offs

,loan_id,month,rung,balance,status
4,L1082,2023-02-01,0,10158.72,current
8,L1112,2023-02-01,0,29208.57,current
11,L1147,2023-02-01,0,24222.34,current
16,L1221,2023-02-01,0,6237.42,current
35,L1555,2023-02-01,0,2885.27,current
...,...,...,...,...,...
36675,L1018,2027-08-01,4,5892.25,120 days late
36705,L1662,2027-08-01,2,2736.99,60 days late
36745,L1662,2027-09-01,3,2736.99,90 days late
36767,L1662,2027-10-01,4,2736.99,120 days late


In [19]:
charge_offs = charge_offs[charge_offs['rung'] == 4].drop_duplicates(subset='loan_id', keep='last')
charge_offs

,loan_id,month,rung,balance,status
906,L1644,2023-07-01,4,2885.27,120 days late
1392,L1030,2023-09-01,4,3830.63,120 days late
3451,L1306,2024-01-01,4,11441.44,120 days late
3822,L2045,2024-01-01,4,14500.35,120 days late
3954,L1082,2024-02-01,4,7769.76,120 days late
...,...,...,...,...,...
36417,L1759,2027-05-01,4,782.89,120 days late
36583,L2069,2027-06-01,4,379.38,120 days late
36609,L1201,2027-07-01,4,3099.20,120 days late
36675,L1018,2027-08-01,4,5892.25,120 days late


In [20]:
len(charged_off)

115

In [21]:
charge_offs = charge_offs.rename(columns = {"month":"charge_off_month","balance":"charge_off_balance"})
charge_offs

,loan_id,charge_off_month,rung,charge_off_balance,status
906,L1644,2023-07-01,4,2885.27,120 days late
1392,L1030,2023-09-01,4,3830.63,120 days late
3451,L1306,2024-01-01,4,11441.44,120 days late
3822,L2045,2024-01-01,4,14500.35,120 days late
3954,L1082,2024-02-01,4,7769.76,120 days late
...,...,...,...,...,...
36417,L1759,2027-05-01,4,782.89,120 days late
36583,L2069,2027-06-01,4,379.38,120 days late
36609,L1201,2027-07-01,4,3099.20,120 days late
36675,L1018,2027-08-01,4,5892.25,120 days late


In [22]:
charge_offs = charge_offs.drop(columns = ['rung','status'])
charge_offs

,loan_id,charge_off_month,charge_off_balance
906,L1644,2023-07-01,2885.27
1392,L1030,2023-09-01,3830.63
3451,L1306,2024-01-01,11441.44
3822,L2045,2024-01-01,14500.35
3954,L1082,2024-02-01,7769.76
...,...,...,...
36417,L1759,2027-05-01,782.89
36583,L2069,2027-06-01,379.38
36609,L1201,2027-07-01,3099.20
36675,L1018,2027-08-01,5892.25


In [23]:
charge_offs = pd.merge(charge_offs,loan_data[['loan_id','grade']], on = "loan_id", how = "left")
charge_offs

,loan_id,charge_off_month,charge_off_balance,grade
0,L1644,2023-07-01,2885.27,E
1,L1030,2023-09-01,3830.63,G
2,L1306,2024-01-01,11441.44,G
3,L2045,2024-01-01,14500.35,F
4,L1082,2024-02-01,7769.76,B
...,...,...,...,...
110,L1759,2027-05-01,782.89,E
111,L2069,2027-06-01,379.38,C
112,L1201,2027-07-01,3099.20,D
113,L1018,2027-08-01,5892.25,A


In [24]:
status_monthly = status_monthly.sort_values(by=['loan_id','month'])
status_monthly

,loan_id,month,rung,balance,status
510,L1001,2023-06-01,0,25191.24,current
748,L1001,2023-07-01,0,24382.48,current
1042,L1001,2023-08-01,0,23573.72,current
1385,L1001,2023-09-01,0,22764.96,current
1787,L1001,2023-10-01,0,21956.20,current
...,...,...,...,...,...
18770,L2224,2025-04-01,0,612.23,current
19935,L2224,2025-05-01,0,449.72,current
21092,L2224,2025-06-01,0,287.21,current
22232,L2224,2025-07-01,0,124.70,current


In [25]:
status_monthly['previous_balance'] = status_monthly.groupby('loan_id')['balance'].shift(1)
status_monthly

,loan_id,month,rung,balance,status,previous_balance
510,L1001,2023-06-01,0,25191.24,current,NaN
748,L1001,2023-07-01,0,24382.48,current,25191.24
1042,L1001,2023-08-01,0,23573.72,current,24382.48
1385,L1001,2023-09-01,0,22764.96,current,23573.72
1787,L1001,2023-10-01,0,21956.20,current,22764.96
...,...,...,...,...,...,...
18770,L2224,2025-04-01,0,612.23,current,774.74
19935,L2224,2025-05-01,0,449.72,current,612.23
21092,L2224,2025-06-01,0,287.21,current,449.72
22232,L2224,2025-07-01,0,124.70,current,287.21


In [26]:
status_monthly['payment_amount'] = status_monthly['previous_balance'] - status_monthly['balance']
status_monthly

,loan_id,month,rung,balance,status,previous_balance,payment_amount
510,L1001,2023-06-01,0,25191.24,current,NaN,NaN
748,L1001,2023-07-01,0,24382.48,current,25191.24,808.76
1042,L1001,2023-08-01,0,23573.72,current,24382.48,808.76
1385,L1001,2023-09-01,0,22764.96,current,23573.72,808.76
1787,L1001,2023-10-01,0,21956.20,current,22764.96,808.76
...,...,...,...,...,...,...,...
18770,L2224,2025-04-01,0,612.23,current,774.74,162.51
19935,L2224,2025-05-01,0,449.72,current,612.23,162.51
21092,L2224,2025-06-01,0,287.21,current,449.72,162.51
22232,L2224,2025-07-01,0,124.70,current,287.21,162.51


In [27]:
payments = status_monthly[status_monthly['payment_amount']>0].copy()
payments

,loan_id,month,rung,balance,status,previous_balance,payment_amount
748,L1001,2023-07-01,0,24382.48,current,25191.24,808.76
1042,L1001,2023-08-01,0,23573.72,current,24382.48,808.76
1385,L1001,2023-09-01,0,22764.96,current,23573.72,808.76
1787,L1001,2023-10-01,0,21956.20,current,22764.96,808.76
2239,L1001,2023-11-01,0,21147.44,current,21956.20,808.76
...,...,...,...,...,...,...,...
18770,L2224,2025-04-01,0,612.23,current,774.74,162.51
19935,L2224,2025-05-01,0,449.72,current,612.23,162.51
21092,L2224,2025-06-01,0,287.21,current,449.72,162.51
22232,L2224,2025-07-01,0,124.70,current,287.21,162.51


In [28]:
payments = payments.drop(columns=['rung','balance','status','previous_balance'])
payments = payments.rename(columns = {'month':'payment_month'})
payments

,loan_id,payment_month,payment_amount
748,L1001,2023-07-01,808.76
1042,L1001,2023-08-01,808.76
1385,L1001,2023-09-01,808.76
1787,L1001,2023-10-01,808.76
2239,L1001,2023-11-01,808.76
...,...,...,...
18770,L2224,2025-04-01,162.51
19935,L2224,2025-05-01,162.51
21092,L2224,2025-06-01,162.51
22232,L2224,2025-07-01,162.51


In [29]:
status_monthly = status_monthly.drop(columns = ['previous_balance','payment_amount'])
status_monthly

,loan_id,month,rung,balance,status
510,L1001,2023-06-01,0,25191.24,current
748,L1001,2023-07-01,0,24382.48,current
1042,L1001,2023-08-01,0,23573.72,current
1385,L1001,2023-09-01,0,22764.96,current
1787,L1001,2023-10-01,0,21956.20,current
...,...,...,...,...,...
18770,L2224,2025-04-01,0,612.23,current
19935,L2224,2025-05-01,0,449.72,current
21092,L2224,2025-06-01,0,287.21,current
22232,L2224,2025-07-01,0,124.70,current


In [30]:
status_monthly.to_csv('/Users/aishwarya/Desktop/Projects/loan-portfolio-risk-analyzer/data/processed/status_monthly.csv', index = False)

In [31]:
payments.to_csv('/Users/aishwarya/Desktop/Projects/loan-portfolio-risk-analyzer/data/processed/payments.csv', index = False)

In [32]:
charge_offs.to_csv('/Users/aishwarya/Desktop/Projects/loan-portfolio-risk-analyzer/data/processed/charge_offs.csv', index = False)